müthiş—senin paylaştığın CSV yapısı **geniş (wide)** formda:

* 1. sütun: `"Item(s)"` → sepetteki ürün sayısı
* Devamı: `"Item 1" ... "Item 32"` → ürün adları (çoğu boş/NaN)

Aşağıya, **yalnızca Apriori** (önerim: `mlxtend`) ile, bu CSV yapısına tam uyumlu **temiz ve anlaşılır** bir akış bıraktım. Kod, sepetleri doğru biçimde çıkarır, Apriori ile sık öğe kümelerini bulur ve kuralları **support / confidence / lift** ile tablolayıp sıralar.

> İstersen en alta **apyori** sürümünü de ekledim; eğitim videonla aynı kütüphane.

---

## A) Apriori (önerilen) — `mlxtend` ile kısa ve net

```python
# pip install mlxtend
import pandas as pd
import numpy as np

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# ========= 1) Veriyi oku =========
df = pd.read_csv("Groceries.csv")  # dosya adını ortamına göre değiştir

# Bu dosya 'Item(s), Item 1, ..., Item 32' formatında
# "Item(s)" sadece sayım, analizde kullanmayacağız
item_cols = [c for c in df.columns if c.strip().lower().startswith("item ")]
assert len(item_cols) > 0, "Item 1..Item N sütunları bulunamadı."

# ========= 2) Sepetleri (transactions) oluştur =========
# Her satır bir sepet: NaN/boşları at, trim yap
transactions = (
    df[item_cols]
    .apply(lambda row: [str(x).strip() for x in row.dropna().tolist() if str(x).strip()], axis=1)
    .tolist()
)

print(f"Toplam sepet: {len(transactions)}")
print("Örnek sepet:", transactions[0][:5], "...")

# (Opsiyonel) Aynı sepette tekrar eden ürünü tekillemek istersen:
# transactions = [list(dict.fromkeys(t)) for t in transactions]  # sırası korunur

# ========= 3) One-hot matrisi (Apriori girdisi) =========
te = TransactionEncoder()
X = te.fit(transactions).transform(transactions)
df_hot = pd.DataFrame(X, columns=te.columns_)  # True/False one-hot

# (Opsiyonel) en sık tekil ürünleri görelim
item_support = df_hot.mean().sort_values(ascending=False)
print("\nEn sık 10 ürün (support):")
print((item_support.head(10)*100).round(2).astype(str) + "%")

# ========= 4) Apriori: sık itemset'ler =========
# Eşikler: veri büyüklüğüne göre ayarla (support ~%0.3 iyi bir başlangıç)
min_support = 0.003      # ~%0.3
max_len     = 2          # 2’li setler (kampanya: bir alana ikinci ... gibi)
min_conf    = 0.20       # 0.2–0.6 arası dene
min_lift    = 3.0        # bağımsızlığa göre 3 kat ve üzeri

freq = apriori(df_hot, min_support=min_support, use_colnames=True, max_len=max_len)
freq = freq.sort_values("support", ascending=False).reset_index(drop=True)
print(f"\nSık itemset sayısı: {len(freq)} (min_support={min_support})")

# ========= 5) Kurallar (support, confidence, lift...) =========
rules = association_rules(freq, metric="confidence", min_threshold=min_conf)

# Filtrele: lift >= min_lift ve sağ taraf boş olmasın
rules = rules[(rules["lift"] >= min_lift) & (rules["consequents"].str.len() > 0)]

# Set'leri okunur hale getir (tuple)
rules["antecedents"] = rules["antecedents"].apply(lambda s: tuple(sorted(list(s))))
rules["consequents"] = rules["consequents"].apply(lambda s: tuple(sorted(list(s))))

# Sırala: önce lift, sonra confidence, support
rules = rules.sort_values(["lift", "confidence", "support"], ascending=False).reset_index(drop=True)

# Görüntüle
cols = ["antecedents", "consequents", "support", "confidence", "lift", "leverage", "conviction"]
print("\nEn iyi 10 kural (lift’e göre):")
print(rules[cols].head(10).to_string(index=False))

# (Opsiyonel) Kaydet
rules[cols].to_csv("apriori_rules_top.csv", index=False)
print("\nKurallar 'apriori_rules_top.csv' dosyasına kaydedildi.")
```

### Parametreyi nasıl seçersin?

* **min\_support**: çok az kural çıkarsa düşür (ör. 0.002); çok fazla gelirse yükselt (ör. 0.01).
* **min\_confidence**: 0.2–0.6 arası dene.
* **min\_lift**: 3 iyi başlangıç; kural azsa 2–2.5’e indir.
* **max\_len**: 2 → basit ve anlaşılır çiftler; 3 yaparsan kombinasyon artar.

---

## B) Aynı şey `apyori` ile (videodaki kütüphaneye yakın)

```python
# pip install apyori
import pandas as pd
from apyori import apriori

# 1) Veriyi oku
df = pd.read_csv("Groceries.csv")
item_cols = [c for c in df.columns if c.strip().lower().startswith("item ")]
assert len(item_cols) > 0

# 2) Sepet listesi hazırla
transactions = (
    df[item_cols]
    .apply(lambda row: [str(x).strip() for x in row.dropna().tolist() if str(x).strip()], axis=1)
    .tolist()
)

print(f"Toplam sepet: {len(transactions)}")

# 3) Apriori (apyori)
min_support = 0.003
min_conf    = 0.20
min_lift    = 3.0
min_len     = 2
max_len     = 2

rule_gen = apriori(
    transactions=transactions,
    min_support=min_support,
    min_confidence=min_conf,
    min_lift=min_lift,
    min_length=min_len,
    max_length=max_len
)
results = list(rule_gen)
print(f"Kural sayısı (apyori): {len(results)}")

# 4) Apyori çıktısını temiz DataFrame’e dönüştür
rows = []
for res in results:
    supp = res.support
    for os in res.ordered_statistics:
        lhs = tuple(os.items_base)
        rhs = tuple(os.items_add)
        conf = os.confidence
        lift = os.lift
        if len(lhs) >= 1 and len(rhs) >= 1:
            rows.append({
                "antecedents": lhs,
                "consequents": rhs,
                "support": supp,
                "confidence": conf,
                "lift": lift
            })

rules_df = pd.DataFrame(rows).sort_values(["lift","confidence","support"], ascending=False).reset_index(drop=True)
print("\nEn iyi 10 kural (lift’e göre):")
print(rules_df.head(10).to_string(index=False))

rules_df.to_csv("apyori_rules_top.csv", index=False)
print("\nKurallar 'apyori_rules_top.csv' dosyasına kaydedildi.")
```

---

### Mini kontrol listesi

* ✅ CSV geniş format: `"Item(s)"`’i **kullanma**, `"Item 1..N"` üzerinden **boşları at**.
* ✅ Aynı sepette tekrar eden ürün varsa **(opsiyonel) tekille**.
* ✅ Apriori girdisi için **TransactionEncoder + one-hot** (mlxtend) ya da doğrudan `apyori` kullan.
* ✅ Kuralları **lift/confidence/support** ile filtrele/sırala.
* ✅ Sonuçları CSV’ye kaydet (raporlamak kolaylaşır).

İstersen buna küçük bir **scatter görselleştirme** (support vs confidence, renk=lift) ya da **en sık 20 ürün çubuğu** da ekleyebilirim.


In [3]:
import pandas as pd
import numpy as np

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [4]:
# ========= 1) Veriyi oku =========
df = pd.read_csv("groceries - groceries.csv")  # dosya adını ortamına göre değiştir

In [5]:
# Bu dosya 'Item(s), Item 1, ..., Item 32' formatında
# "Item(s)" sadece sayım, analizde kullanmayacağız
item_cols = [c for c in df.columns if c.strip().lower().startswith("item ")]
assert len(item_cols) > 0, "Item 1..Item N sütunları bulunamadı."

In [6]:
# ========= 2) Sepetleri (transactions) oluştur =========
# Her satır bir sepet: NaN/boşları at, trim yap
transactions = (
    df[item_cols]
    .apply(lambda row: [str(x).strip() for x in row.dropna().tolist() if str(x).strip()], axis=1)
    .tolist()
)

In [7]:
print(f"Toplam sepet: {len(transactions)}")
print("Örnek sepet:", transactions[0][:5], "...")

Toplam sepet: 9835
Örnek sepet: ['citrus fruit', 'semi-finished bread', 'margarine', 'ready soups'] ...


In [8]:
# (Opsiyonel) Aynı sepette tekrar eden ürünü tekillemek istersen:
# transactions = [list(dict.fromkeys(t)) for t in transactions]  # sırası korunur

In [9]:
# ========= 3) One-hot matrisi (Apriori girdisi) =========
te = TransactionEncoder()
X = te.fit(transactions).transform(transactions)
df_hot = pd.DataFrame(X, columns=te.columns_)  # True/False one-hot

In [10]:
# (Opsiyonel) en sık tekil ürünleri görelim
item_support = df_hot.mean().sort_values(ascending=False)
print("\nEn sık 10 ürün (support):")
print((item_support.head(10)*100).round(2).astype(str) + "%")


En sık 10 ürün (support):
whole milk          25.55%
other vegetables    19.35%
rolls/buns          18.39%
soda                17.44%
yogurt              13.95%
bottled water       11.05%
root vegetables      10.9%
tropical fruit      10.49%
shopping bags        9.85%
sausage               9.4%
dtype: object


In [11]:
# ========= 4) Apriori: sık itemset'ler =========
# Eşikler: veri büyüklüğüne göre ayarla (support ~%0.3 iyi bir başlangıç)
min_support = 0.003      # ~%0.3
max_len     = 2          # 2’li setler (kampanya: bir alana ikinci ... gibi)
min_conf    = 0.20       # 0.2–0.6 arası dene
min_lift    = 3.0        # bağımsızlığa göre 3 kat ve üzeri

In [12]:
freq = apriori(df_hot, min_support=min_support, use_colnames=True, max_len=max_len)
freq = freq.sort_values("support", ascending=False).reset_index(drop=True)
print(f"\nSık itemset sayısı: {len(freq)} (min_support={min_support})")


Sık itemset sayısı: 1276 (min_support=0.003)


In [13]:
# ========= 5) Kurallar (support, confidence, lift...) =========
rules = association_rules(freq, metric="confidence", min_threshold=min_conf)

In [14]:
# Filtrele: lift >= min_lift ve sağ taraf boş olmasın
rules = rules[(rules["lift"] >= min_lift) & (rules["consequents"].str.len() > 0)]


In [15]:
# Set'leri okunur hale getir (tuple)
rules["antecedents"] = rules["antecedents"].apply(lambda s: tuple(sorted(list(s))))
rules["consequents"] = rules["consequents"].apply(lambda s: tuple(sorted(list(s))))

In [16]:
# Sırala: önce lift, sonra confidence, support
rules = rules.sort_values(["lift", "confidence", "support"], ascending=False).reset_index(drop=True)


In [17]:
# Görüntüle
cols = ["antecedents", "consequents", "support", "confidence", "lift", "leverage", "conviction"]
print("\nEn iyi 10 kural (lift’e göre):")
print(rules[cols].head(10).to_string(index=False))


En iyi 10 kural (lift’e göre):
             antecedents           consequents  support  confidence      lift  leverage  conviction
(Instant food products,)     (hamburger meat,) 0.003050    0.379747 11.421438  0.002783    1.558640
                (flour,)              (sugar,) 0.004982    0.286550  8.463112  0.004394    1.354182
     (processed cheese,)        (white bread,) 0.004169    0.251534  5.975445  0.003471    1.279824
               (liquor,)       (bottled beer,) 0.004677    0.422018  5.240594  0.003785    1.590831
                (herbs,)    (root vegetables,) 0.007016    0.431250  3.956477  0.005243    1.566596
              (berries,) (whipped/sour cream,) 0.009049    0.272171  3.796886  0.006666    1.275461
                 (rice,)    (root vegetables,) 0.003152    0.413333  3.792102  0.002321    1.518753
                (flour,)          (margarine,) 0.003762    0.216374  3.694516  0.002744    1.201382
        (baking powder,) (whipped/sour cream,) 0.004575    0.258621 

In [18]:
# (Opsiyonel) Kaydet
rules[cols].to_csv("apriori_rules_top.csv", index=False)
print("\nKurallar 'apriori_rules_top.csv' dosyasına kaydedildi.")


Kurallar 'apriori_rules_top.csv' dosyasına kaydedildi.


süper—**mlxtend ile Apriori** akışını doğru kurmuşsun. Aşağıda yaptığın her adımı “ne, neden, nasıl” diye pekiştireyim; ardından da **ilk 10 kuralı nasıl yorumlayacağını** anlatayım (ve istersen koşturabileceğin mini bir “yorumlayıcı” kod da verdim).

---

# Adım adım ne yaptın ve neden

## 1) Sepetleri (transactions) oluşturma

* **Ne?** Geniş (Item 1…Item N) formatındaki her satırı bir alışveriş sepetine çevirip, boşları/NaN’ları attın.
* **Neden?** Apriori itemset’ler üzerinde çalışır; önce her satır bir **ürün listesi** (transaction) olmalı.
* **İpucu:** Aynı sepette aynı ürün iki kez varsa tekillemek istersen:

  ```python
  transactions = [list(dict.fromkeys(t)) for t in transactions]
  ```

## 2) One-Hot matrisi

* **Ne?** `TransactionEncoder` ile `transactions → df_hot (True/False)` matrisine dönüştürdün.
* **Neden?** `mlxtend.frequent_patterns.apriori` bu formatı bekler. Her sütun bir ürün; her satır bir sepet.

## 3) Apriori (sık öğe kümeleri)

* **Ne?**

  ```python
  freq = apriori(df_hot, min_support=..., use_colnames=True, max_len=2)
  ```

  ile support eşiğini geçen **sık itemset**’leri buldun.
* **Neden?** Kurallar bu sık kümelerden türetilir. `max_len=2` diyerek “ikili kombinasyonlar”a odaklandın (kampanya mantığı için çok pratik).
* **Parametreler:**

  * `min_support`: Veri büyüklüğüne bağlı kritik eşik. Çok az kural gelirse düşür, çok fazla gelirse yükselt.
  * `max_len`: 2 tutmak kuralları **okunur** tutar; 3 yaparsan çok artar.

## 4) Kuralların çıkarılması

* **Ne?**

  ```python
  rules = association_rules(freq, metric="confidence", min_threshold=min_conf)
  ```

  sonra `rules`’u `lift ≥ min_lift` ile filtreledin ve sıraladın.
* **Neden?**

  * **confidence** = Koşullu olasılık $P(B|A)$ → “A varsa B de geliyor mu?”
  * **lift** = $\frac{P(B|A)}{P(B)}$ → bağımsızlığa göre **kaç kat** daha sık birlikte?

    * **>1**: pozitif ilişki, **≈1**: bağımsız, **<1**: olumsuz ilişki.

## 5) Sıralama ve çıktı

* **Ne?** Kuralları `lift → confidence → support` ile sıralayıp tablolaştırdın/kaydettin.
* **Neden?** Lift birlikteliğin “göreli gücünü”, support ise “**kaç sepette** görüldüğünü” söyler; ikisini birlikte görmek gerekir.

---

# İlk 10 kuralı nasıl yorumlarsın?

Her satırda şu sütunlar olur (mlxtend):

* **antecedents**: Sol taraf (A) — tetikleyen ürün(ler)
* **consequents**: Sağ taraf (B) — A olduğunda birlikte gelen ürün(ler)
* **support**: $P(A∪B)$. Toplam sepetlerin **yüzdesi**.

  * **Adet** olarak görmek istersen: `count = round(support * N)`, N = toplam sepet.
* **confidence**: $P(B|A)$. A olan sepetlerin yüzde kaçı B de almış?
* **lift**: $\frac{P(B|A)}{P(B)}$. **>1** ise A varsa B’nin gelme olasılığı artıyor; **mesela lift=3 → 3 kat**.
* **leverage**: $P(A∪B) - P(A)P(B)$. “Beklenenden **ne kadar fazla** birlikte geliyorlar?” Mutlak fark, istersen adete çevirebilirsin: `leverage * N`.
* **conviction**: (1−P(B)) / (1−P(B|A)). “A → B” ifadesinin tek yönlü gücünü ölçer (yüksek daha iyi).

## 10 kurallık bir liste için pratik okuma rehberi

Her kural için aşağıdakileri **3–4 satırda** düşün:

1. **İş anlamı:** “A alanların %X’i B de alıyor (confidence). Bu, B’nin genel alım oranının (P(B)) lift katı.”
2. **Hacim:** “Destek %s → yaklaşık **s** sepette (support \* N) gözlenmiş.”
3. **Eylem:** “A yakınında B’yi konumlandırma, birlikte kampanya/bundle, çapraz satış önerisi.”
4. **Sağlama:** “Support çok düşük mü? (operasyonal değer). Lift çok yüksek ama support çok küçükse ‘niş kural’ olabilir.”

> **Örnek yorum şablonu** (rakamlar hayali; senin tablodan okuyacaksın):
>
> * Kural: **{whole milk} → {yogurt}**
>
>   * support = 0.012 ⇒ \~%1.2; **≈ 118 sepet** (N=9 835 varsayalım)
>   * confidence = 0.48 ⇒ “whole milk alanların **%48’i** yogurt da almış”
>   * P(yogurt) = 0.20 ise ⇒ **lift = 0.48 / 0.20 = 2.4** (2.4 kat fazla)
>   * leverage = 0.012 − 0.20 \* P(whole milk) ⇒ “**beklenenden** şu kadar fazla birliktelik”
>   * **Aksiyon:** Soğuk raflarında birlikte konumlandırma; “süt alana yoğurt %10” kampanyası.

### Dikkat edilecekler

* **Yalnızca lift’e bakma.** Lift yüksek ama **support** çok düşükse iş değeri sınırlı olabilir.
* “**Genelci ürünler**” (örn. su, ekmek, whole milk) çok sık görülür; lift ≈ 1 ise **bilgi değeri düşüktür**.
* **Aynı ilişkili çiftin iki yönü** (A→B ve B→A) farklı confidence üretir; kararını **iş senaryosuna** göre ver (hangi tarafı “öneri” yapmak istiyorsun?).
* **Çakışan/tekrarlı kurallar**: `({A,B}→{C})` ile `({A}→{B,C})` benzer olabilir; gereksiz tekrarları filtrelemek isteyebilirsin.

---

# (İsteğe bağlı) Kuralları otomatik “insan diliyle” özetleyen mini yardımcı

Aşağıyı **kendi ortamında** çalıştırırsan, ilk 10 kuralı **adetlere çevirip** kısa cümlelerle raporlar:

```python
def explain_rules(rules_df, df_hot, transactions, top=10):
    N = len(transactions)
    # Tekil ürün support'ları (P(item))
    single_support = df_hot.mean()

    out = []
    for i, row in rules_df.head(top).iterrows():
        A = row["antecedents"]; B = row["consequents"]
        supp = row["support"]; conf = row["confidence"]; lift = row["lift"]
        # P(B): eğer tek ürünse doğrudan al; birden fazlaysa yaklaşıksal olarak min tekil support'u yazalım (tamı itemset support'u ister)
        if len(B) == 1:
            pB = float(single_support[list(B)[0]])
        else:
            pB = np.nan

        out.append({
            "rule": f"{A} -> {B}",
            "support(%)": round(supp*100, 2),
            "support_count": int(round(supp*N)),
            "confidence(%)": round(conf*100, 2),
            "lift": round(lift, 3),
            "baseline_P(B)(%)": round(pB*100, 2) if not np.isnan(pB) else None
        })

    return pd.DataFrame(out)

# Örnek kullanım:
# summary_df = explain_rules(rules, df_hot, transactions, top=10)
# print(summary_df.to_string(index=False))
```

> Not: `P(B)`’yi çok ürünlü consequents için tam hesaplamak istersen, `freq` içindeki itemset support’larını bir sözlüğe çevirip oradan çekebilirsin:

```python
# itemset -> support haritası
freq_map = {tuple(sorted(list(s))): sup for s, sup in zip(freq["itemsets"], freq["support"])}

def get_itemset_support(items):
    key = tuple(sorted(list(items)))
    return freq_map.get(key, np.nan)
```

---

# “İyi kural” için hızlı kontrol listesi

* **Support (adet)**: Operasyonel karşılığı var mı? (örn. ≥ 50–100 sepet)
* **Confidence**: En az %30–40? (domaine göre değişir)
* **Lift**: ≥ 1.5–2 genelde anlamlı; ≥ 3 çok güçlü (dikkat: support da makul olsun)
* **Leverage**: Pozitif ve anlamlı büyüklükte mi? (örn. ≥ +0.005 ⇒ N=10k ise \~50 sepet “ekstra birliktelik”)
* **İş mantığı**: Kampanya/bundle/raf yerleşimine **gerçekten** uyuyor mu?

---

## Sonraki adımlar (istersen)

* `max_len=3` ile üçlü kombinasyonları dene; `min_support`’u biraz yükselt.
* “Genelci ürünleri” dışlamak için ya **min\_support**’u artır ya da belirli ürünleri filtrele.
* **Support–Confidence scatter (renk=lift)** grafiğiyle ısı haritası çıkar.
* Segment bazında Apriori (ör. hafta içi / hafta sonu, sabah / akşam).

İstersen, senin ürettiğin `rules` DataFrame’ini bana (ilk 10 satırı) yapıştır; aynı şablonla **tek tek** yorumlayıp “hangi kampanyaya nasıl çevrilir” diye kısa aksiyon önerileri de çıkarayım.
